# IELTS Writing Tutor — Finetune Qwen2.5 3B với HuggingFace TRL + PEFT

**Dataset:** `nlpatunt/D_Ielts_Writing_Task_2_Dataset`  
**Base model:** `Qwen/Qwen2.5-3B-Instruct`
**Method:** SFT + LoRA (QLoRA 4-bit)  
**GPU:** Kaggle T4 x2 (`device_map=auto` tự phân bổ 2 GPU)

> ⚠️ Settings → Accelerator → **GPU T4 x2**  
> ⚠️ Cần HuggingFace token để download Llama 3.1 (gated model)

## 1. Cài đặt dependencies

In [ ]:
#%capture
#!pip install -q transformers==4.47.0
#!pip install -q trl==0.12.0
#!pip install -q peft==0.14.0
#!pip install -q bitsandbytes>=0.46.1
#!pip install -q accelerate==1.2.1
#!pip install -q datasets

## 3. Imports

In [ ]:
!pip install -q --upgrade transformers
!pip install -q --upgrade accelerate

In [ ]:
!pip install -q bitsandbytes --upgrade

In [ ]:
import os
import bitsandbytes
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR   = "/kaggle/working/ielts-qwen"

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_properties(i).name
    vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} — {vram:.1f} GB")

## 4. Load model với QLoRA (4-bit)

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — tốt nhất cho LLM
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # double quantization tiết kiệm thêm VRAM
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",                  # tự phân bổ layers lên 2 GPU
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded!")
print(f"Model dtype: {model.dtype}")
print(f"Device map: {model.hf_device_map}")

## 5. Chuẩn bị model cho QLoRA training

In [ ]:
# Bước bắt buộc trước khi gắn LoRA khi dùng 4-bit
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Load và format dataset

In [ ]:
raw_dataset = load_dataset("nlpatunt/D_Ielts_Writing_Task_2_Dataset")
print(raw_dataset)

In [ ]:
SYSTEM_PROMPT = """You are an expert IELTS Writing examiner with 10+ years of experience.
Your task is to evaluate IELTS Writing Task 2 essays according to the official 4 criteria:
- Task Achievement (TA): How well the essay addresses the prompt
- Coherence and Cohesion (CC): Organization, flow, and linking
- Lexical Resource (LR): Vocabulary range and accuracy
- Grammatical Range and Accuracy (GRA): Grammar variety and correctness

Provide detailed feedback for each criterion, an overall band score, and actionable improvement suggestions."""


def format_sample(row):
    user_message = f"""Please evaluate the following IELTS Writing Task 2 essay.

**Prompt:** {row['prompt']}

**Essay:**
{row['essay']}"""

    assistant_message = f"**Overall Band Score: {row['band_score']}**\n\n{row['evaluation']}"

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = raw_dataset["train"].map(format_sample, remove_columns=raw_dataset["train"].column_names)
val_dataset   = raw_dataset["validation"].map(format_sample, remove_columns=raw_dataset["validation"].column_names)
test_dataset  = raw_dataset["test"].map(format_sample, remove_columns=raw_dataset["test"].column_names)

# Lấy subset nhỏ để chạy thử
train_dataset = train_dataset.select(range(3000))
val_dataset   = val_dataset.select(range(200))

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("\nSample:")
print(train_dataset[0]["text"][:400], "...")

## 7. VRAM check

In [ ]:
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved  = torch.cuda.memory_reserved(i) / 1024**3
    total     = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | Allocated: {allocated:.2f}GB | Reserved: {reserved:.2f}GB | Total: {total:.2f}GB | Free: {total - reserved:.2f}GB")

## 8. Training callbacks

In [ ]:
class TrainingMonitorCallback(TrainerCallback):
    """Print loss + VRAM usage mỗi logging_steps"""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step     = state.global_step
        loss     = logs.get("loss", "N/A")
        eval_loss= logs.get("eval_loss", None)
        lr       = logs.get("learning_rate", "N/A")

        vram_info = ""
        for i in range(torch.cuda.device_count()):
            used = torch.cuda.memory_reserved(i) / 1024**3
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            vram_info += f" GPU{i}: {used:.1f}/{total:.1f}GB"

        msg = f"Step {step:4d} | loss: {loss} | lr: {lr}"
        if eval_loss:
            msg += f" | eval_loss: {eval_loss}"
        msg += f" |{vram_info}"
        print(msg)

## 9. SFT Trainer config

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[TrainingMonitorCallback()],
    args=SFTConfig(
        # --- Core ---
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,     # effective batch = 16
        warmup_ratio=0.05,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",

        # --- Precision ---
        fp16=True,
        bf16=False,

        # --- Sequence ---
        max_seq_length=512,
        dataset_text_field="text",
        packing=True,

        # --- Optimization ---
        optim="paged_adamw_8bit",          # paged optimizer cho QLoRA
        weight_decay=0.01,
        gradient_checkpointing=True,       # tiết kiệm VRAM
        gradient_checkpointing_kwargs={"use_reentrant": False},

        # --- Logging & saving ---
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        output_dir=OUTPUT_DIR,

        seed=42,
        report_to="none",
    ),
)

print("Trainer ready!")

## 10. Training

In [ ]:
print("Starting training...")
trainer_stats = trainer.train()

print(f"\n=== Training Complete ===")
print(f"Time: {trainer_stats.metrics['train_runtime']:.1f}s ({trainer_stats.metrics['train_runtime']/3600:.2f}h)")
print(f"Steps: {trainer_stats.metrics['train_steps_per_second']:.2f} steps/s")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

## 11. Quick inference test

In [ ]:
model.eval()

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": f"Please evaluate the following IELTS Writing Task 2 essay.\n\n**Prompt:** {TEST_PROMPT}\n\n**Essay:**\n{TEST_ESSAY}"},
]

# Fix: tokenize=False trước, rồi tokenize riêng
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],           # ← truyền input_ids trực tiếp
        attention_mask=inputs["attention_mask"], # ← thêm attention_mask
        max_new_tokens=1024,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print("=== MODEL OUTPUT ===")
print(response)

## 12. Lưu LoRA adapter

In [ ]:
ADAPTER_DIR = "/kaggle/working/ielts-lora-adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
size = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR))
print(f"Adapter saved to {ADAPTER_DIR}")
print(f"Total size: {size / 1024**2:.1f} MB")

## 13. Merge LoRA + Export GGUF cho Ollama

> Bước này merge adapter vào base model và convert sang GGUF  
> Cần ~15GB RAM — chạy sau khi training xong

In [ ]:
%%capture
!pip install -q llama-cpp-python
!git clone https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!pip install -q gguf sentencepiece protobuf

In [ ]:
# Cell 1: uninstall conflict trước
!pip uninstall -y torchao llama-cpp-python transformers

In [ ]:
!pip install -q gguf
!pip install -q sentencepiece
!pip install -q protobuf

In [ ]:
!pip install -q transformers --upgrade

In [ ]:
!pip uninstall -y torchao

In [2]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

MERGED_DIR = "/kaggle/working/ielts-merged"
GGUF_DIR   = "/kaggle/working/ielts-gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

# Load base model ở fp16 để merge (không 4-bit)
print("Loading base model for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",           # merge trên CPU để đủ RAM
    trust_remote_code=True,
)

# Merge LoRA vào base
print("Merging LoRA adapter...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

# Lưu merged model
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

# Convert sang GGUF Q4_K_M
print("Converting to GGUF...")
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_DIR}/ielts-llama3.1-8b-q4_k_m.gguf --outtype q4_k_m
print("GGUF exported!")

RuntimeError: Failed to import transformers.models.bloom.modeling_bloom because of the following error (look up to see its traceback):
partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

## 14. Dùng GGUF với Ollama (local)

Sau khi download file GGUF về máy:

```bash
# Tạo Modelfile
cat > Modelfile << 'EOF'
FROM ./ielts-llama3.1-8b-q4_k_m.gguf

SYSTEM """You are an expert IELTS Writing examiner with 10+ years of experience.
Evaluate essays according to TA, CC, LR, GRA criteria.
Provide detailed feedback and actionable improvement suggestions."""

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 2048
EOF

# Load vào Ollama
ollama create ielts-tutor -f Modelfile

# Test
ollama run ielts-tutor
```